# Sanity check: training a classifier on verb classes using pre-extracted features

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, recall_score, precision_score, classification_report

df_train = pd.read_csv('../data/annotations/processed/train.csv').set_index('narration_id')
df_val = pd.read_csv('../data/annotations/processed/val.csv').set_index('narration_id')

train_data = torch.load('../data/features/Verbs/egovlp_plus/features_train.pt')
val_data = torch.load('../data/features/Verbs/egovlp_plus/features_val.pt')

X_train, y_train = [], []
for n_id, tensors in train_data.items():
    if n_id in df_train.index:
        X_train.append(tensors['video'].numpy()) 
        y_train.append(df_train.loc[n_id, 'verb_class']) 

X_val, y_val = [], []
for n_id, tensors in val_data.items():
    if n_id in df_val.index:
        X_val.append(tensors['video'].numpy())
        y_val.append(df_val.loc[n_id, 'verb_class'])

X_train, y_train = np.vstack(X_train), np.array(y_train)
X_val, y_val = np.vstack(X_val), np.array(y_val)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}")



In [ ]:
len(np.unique(y_train))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Training logistic regressor...")
clf = LogisticRegression(max_iter=2000, n_jobs=-1) 
#clf = LinearSVC(dual=False, max_iter=1000)
clf.fit(X_train_scaled, y_train)

print("Evaluation on validation set...")
preds = clf.predict(X_val_scaled)
score = classification_report(y_val, preds,zero_division=0)
print(score)